In [ ]:
#Все библиотеки тут:
#для парсинга
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
import random
import csv
import numpy as np

#для препроцессинга
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

#для добавления запросов
import json
import requests
from typing import List, Dict
import uuid
import google.generativeai as genai
from typing import List, Dict
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig
import torch
import random

#для обучения

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.utils import resample
from scipy.stats import mode
from scipy.sparse import issparse
from sklearn.naive_bayes import (MultinomialNB, ComplementNB)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
from sentence_transformers import SentenceTransformer, util
import torch


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
books_pn=pd.read_csv('books_pn_text.csv', encoding='utf-8')

In [ ]:
df_clean = books_pn[~books_pn['clean annotation'].str.contains('открывается в программе adobe reader', case=False, na=False)]

In [ ]:
df_clean['text'] = df_clean['text'].astype(str)

In [ ]:
train_df, test_df = train_test_split( #определим, что у нас есть материал для для трейна и для теста
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['label']
)

In [ ]:
#определяем Х и у
X=df_clean['text'].tolist()
y=df_clean['label'].values #таргет уже разбит на бинарник (1, 0), не нуждается в доп кодировании

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #делим на выборки

In [ ]:
#вместо обычных векторов, создадим эмбеддинги

In [ ]:
model_name = "sergeyzh/rubert-mini-frida" #очень хотелось попробовать фриду, тк на нее очень хорошие отзывы среди русскоязычных моделей
embedder = SentenceTransformer(model_name)


embedder = embedder.to('cuda')


print(f"   Модель загружена: {model_name}")


modules.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/712 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.41M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/732 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/196 [00:00<?, ?B/s]

   Модель загружена: sergeyzh/rubert-mini-frida


In [ ]:
#создание эмбеддингов для классификации
X_train_emb = embedder.encode(
    X_train,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

X_test_emb = embedder.encode(
    X_test,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_emb)
X_test_sc = scaler.transform(X_test_emb)

In [ ]:
model_lr=LogisticRegression(random_state=42, max_iter=1000)

model_lr.fit(X_train_sc, y_train)

y_predict2=model_lr.predict(X_test_sc)

In [ ]:
print('Логистическая регрессия с эмбеддингами')
print(classification_report(y_test, y_predict2))

Логистическая регрессия с эмбеддингами
              precision    recall  f1-score   support

           0       0.59      0.53      0.56       160
           1       0.57      0.63      0.60       157

    accuracy                           0.58       317
   macro avg       0.58      0.58      0.58       317
weighted avg       0.58      0.58      0.58       317



In [ ]:
print(" Проверка модели:")
print(f"   Тип: {type(model_lr)}")
print(f"   Есть веса: {hasattr(model_lr, 'coef_')}")
if hasattr(model_lr, 'coef_'):
    print(f"   Веса: {model_lr.coef_.shape}")
    print(f"   Средний вес: {model_lr.coef_.mean():.4f}")
    print(f"   Интерсепт: {model_lr.intercept_[0]:.4f}")

 Проверка модели:
   Тип: <class 'sklearn.linear_model._logistic.LogisticRegression'>
   Есть веса: True
   Веса: (1, 312)
   Средний вес: -0.0434
   Интерсепт: -0.0713


In [ ]:
books_data = pd.read_csv('books_id.csv', encoding='utf-8')  #база книг для поиска

In [ ]:
def clean_text(text, language='russian'):
    text = text. lower() #приводим к нижнему регистру
    text = re.sub(r'http\S+|www\S+', ' ', text) #удаляем все похожее на ссылку
    text = re.sub(r'[^a-zа-яё\s]', '', text)  #удаляем все символы кроме букв
    text = re. sub(r'\s+', ' ', text) #убираем лишние пробелы
    tokens=text.split() #токенизируем
    stop_words = set(stopwords.words('russian')) #убираем стоп слова
    return ' '.join(tokens)

In [ ]:
def rank_books(query, model, embedder, scaler, df_books, top_k=3):
    """
    Ранжирует все книги по релевантности запросу + используем logprob
    """
    print(f"\n ПОИСК: '{query}'")

    # Очищаем запрос
    query_clean = clean_text(query)

    results = []
    total_books = len(df_books)

    # Создаем текст запроса с префиксом
    query_text = 'query: ' + query_clean + ' doc: '

    for idx, book in df_books.iterrows():
        # Очищаем аннотацию
        annotation_clean = clean_text(book['annotation'])
        # Формируем полный текст (запрос + аннотация)
        text = query_text + annotation_clean

        # Создаем эмбеддинг
        emb = embedder.encode([text], convert_to_numpy=True, normalize_embeddings=True)

        # Масштабируем
        emb_scaled = scaler.transform(emb)

        # Получаем вероятность релевантности
        #score = model.predict_proba(emb_scaled)[0, 1]
        prob_class_0, prob_class_1 = model.predict_proba(emb_scaled)[0]
        # Берём вероятность класса "релевантно" (label=1)
        prob = prob_class_1

        # Вычисляем logprob (с защитой от log(0))
        logprob = np.log(prob + 1e-10)

        results.append({
            'book_id': book['book_id'],
            'title': book['title'],
            'annotation': book['annotation'],
            'link': book.get('link', ''),
            'prob_class_0': prob_class_0,
            'prob_class_1': prob_class_1,
            'score': prob_class_1,
            'logprob': logprob
            #'score': float(score)
            })


    # Сортируем по logprob (чем ближе к 0, тем лучше)
    results.sort(key=lambda x: x['logprob'], reverse=True)
    # Сортируем по убыванию оценки
    #results.sort(key=lambda x: x['score'], reverse=True)

    # Добавляем ранг
    #for i, result in enumerate(results, 1):
        #result['rank'] = i

    return results[:top_k]


In [ ]:
print(books_data.columns.tolist())

['title', 'annotation', 'link', 'book_id']


In [ ]:
# Проверьте, сколько книг знает модель
unique_books_in_train = train_df['book_id'].nunique()
print(f"Модель знает {unique_books_in_train} книг")

# Убедитесь, что вы ищете только по ним
df_books_for_search = books_data[books_data['book_id'].isin(train_df['book_id'].unique())]
print(f"Будет искать среди {len(df_books_for_search)} книг")

Модель знает 944 книг
Будет искать среди 944 книг


In [ ]:
#ТЕСТ
test_queries = [
    "психология",
]

# Запускаем поиск для каждого запроса
for query in test_queries:
    results = rank_books(
        query,
        model_lr,
        embedder,
        scaler,
        df_books_for_search,
        top_k=3
    )

    print(f"\n ТОП-3 КНИГ ДЛЯ '{query}':")

    for i, result in enumerate(results, 1):
        print(f"\n{i}. {result['title']}")
        print(f"   Оценка: {result['score']:.2%}")
        print(f"   Logprob (уверенность): {result['logprob']:.3f}")
        print(f"   Аннотация: {result['annotation'][:120]}...")
        if result['link']:
            print(f"   Ссылка: {result['link']}")


 ПОИСК: 'психология'

 ТОП-3 КНИГ ДЛЯ 'психология':

1. Ошибка
   Оценка: 99.99%
   Logprob (уверенность): -0.000
   Аннотация: Ошибка: HTTPSConnectionPool(host='bombora.ru', port=443): Read timed out. (read timeout=20)...
   Ссылка: https://bombora.ru/book/90325/

2. Ошибка
   Оценка: 99.99%
   Logprob (уверенность): -0.000
   Аннотация: Ошибка: HTTPSConnectionPool(host='bombora.ru', port=443): Read timed out. (read timeout=20)...
   Ссылка: https://bombora.ru/book/88274/

3. Природные геропротекторы. Анализ 25 природных соединений, способных замедлить процессы старения и улучшить качество жизни
   Оценка: 99.99%
   Logprob (уверенность): -0.000
   Аннотация: Данная книга представляет собой обзор потенциальных природных геропротекторов, включая такие вещества, как уролитин A, а...
   Ссылка: https://bombora.ru/book/186883/


In [ ]:
#СЕМАНТИЧЕСКИЙ ПОИСК

In [ ]:
all_books=pd.read_csv('books_id.csv', encoding='utf-8')

In [ ]:
all_books.columns.tolist()

['title', 'annotation', 'link', 'book_id']

In [ ]:
all_books['annotation'] = all_books['annotation'].fillna('').astype(str)
all_books['title'] = all_books['title'].fillna('').astype(str)
all_books['link'] = all_books.get('link', '').fillna('').astype(str)

In [ ]:
#препроцессинг
all_books['title_clean'] = all_books['title'].apply(clean_text)
all_books['annotation_clean'] = all_books['annotation'].apply(clean_text)

# Создаём текст для эмбеддинга (название + аннотация)
all_books['text_for_embedding'] = all_books['title_clean'] + ' ' + all_books['annotation_clean']

print(f"   Пример текста для эмбеддинга:")
print(f"   {all_books['text_for_embedding'].iloc[0][:200]}...")


   Пример текста для эмбеддинга:
   любовь которая убивает как распознать психологическое насилие и построить здоровые отношения эта книга для вас если в отношениях вы чувствуете себя плохо но не знаете почему ваш партнер позволяет себе...


In [ ]:
#создаем эмбеддинги, использую в качестве модели все ту же фриду
book_embeddings = embedder.encode(
    all_books['text_for_embedding'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print(f"\n Эмбеддинги созданы")
print(f"   Количество: {book_embeddings.shape[0]}")
print(f"   Размерность: {book_embeddings.shape[1]}")

Batches:   0%|          | 0/553 [00:00<?, ?it/s]


 Эмбеддинги созданы
   Количество: 17687
   Размерность: 312


In [ ]:
book_embeddings = book_embeddings.to('cuda')


In [ ]:
def semantic_search(query, embedder, book_embeddings, df_books, top_k=3):
    print(f"\n ПОИСК: '{query}'")

    query_clean = clean_text(query)
    query_emb = embedder.encode(query_clean, convert_to_tensor=True, normalize_embeddings=True) #добавляем создание эмбеддингов запроса именно через функцию, тк они могут меняться от пользователя к пользователю

    # Косинусное сходство
    similarities = util.cos_sim(query_emb, book_embeddings)[0]

    top_indices = similarities.topk(top_k).indices
    top_scores = similarities.topk(top_k).values
    #на cpu только для чтения данных
    top_indices = top_indices.cpu().tolist()
    top_scores = top_scores.cpu().tolist()

    results = []
    for idx, score in zip(top_indices, top_scores):
        results.append({
            'book_id': all_books.iloc[idx]['book_id'],
            'title': all_books.iloc[idx]['title'],
            'annotation': all_books.iloc[idx]['annotation'],
            'link': all_books.iloc[idx].get('link', ''),
            'similarity': float(score)
        })


    return results


In [ ]:
# Тест
results = semantic_search("книги по психологии", embedder, book_embeddings, all_books, top_k=3)

for r in results:
    print(f"\n{r['title']}")
    print(f"   Сходство: {r['similarity']:.2%}")
    print(f"   {r['annotation'][:100]}...")


 ПОИСК: 'книги по психологии'

Психология. Все, что вам нужно знать, - в одной книге
   Сходство: 70.34%
   В этой книге Алан Портер, преподаватель психологии Вестминстерского университета, емко и доступно из...

Психология. Все, что вам нужно знать, - в одной книге
   Сходство: 70.34%
   В этой книге Алан Портер, преподаватель психологии Вестминстерского университета, емко и доступно из...

50 великих книг по психологии
   Сходство: 64.83%
   НЕЗАКОННОЕ ПОТРЕБЛЕНИЕ НАРКОТИЧЕСКИХ СРЕДСТВ, ПСИХОТРОПНЫХ ВЕЩЕСТВ, ИХ АНАЛОГОВ ПРИЧИНЯЕТ ВРЕД ЗДОРО...


In [ ]:
нафиг делала обучения не ясно
эмбеддинги это очень классно, но семантический поиск элементарно спарвляется с этой задачей

In [ ]:
import gradio as gr

In [ ]:
iface_sentiment = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(label="Введите запрос", placeholder="Например: 'книги про еду'"),
    outputs=gr.Textbox(label="Результаты"),
    title="Семантический поиск",
    description="Введите текст, чтобы получить книгу под настроение"
)

iface_sentiment.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1220: UserWarning: Expected at least 4 arguments for function <function semantic_search at 0x79e18e257c40>, received 1.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://312a9930db7df4f31d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
